In [ ]:
import pandas as pd
import numpy as np
import os
import itertools as it
from snp_analysis_tools_sherlock import *
from coalescence_analysis_tools import *
from plot_tools import *
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:


fname = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/old_species/species/metadata.tsv'
df_metadata= pd.read_csv(fname, delimiter = '\t')
df_metadata

def transform_df(df_abundance):
    df_abundance['Lineage'] = df_abundance['species_id'].transform(lambda x: df_metadata.loc[df_metadata['species_id'] == x,'Lineage'].values[0])
    df_abundance['species'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-1])
    df_abundance['genus'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-2])
    df_abundance['family'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-3])
    df_abundance['phyla'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[1])
    return df_abundance

df_metadata  =transform_df(df_metadata)
df_metadata

In [ ]:
all_dfs=[]
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
e003_metadata = pd.read_csv('e003_coalescence_metadata_round4_good.csv').set_index('sample')
for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
     #   try:
        fname = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{sp}/{ino}_parent1_info.csv'
        if os.path.exists(fname):
            df_both=get_both_dfs(fname)#.set_index('sample')
       # except:
        #    continue
            df_both['species_id']=sp
            df_both['sp_plot']=df_metadata.loc[df_metadata['species_id']==int(sp),'species'].values[0]
         #   print(df_metadata.loc[df_metadata['species_id']==int(sp),'species'])
            
            df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
            df_both=df_both.loc[df_both['total_shift']<.1,:]
        
            df_both_meta = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                                           e003_metadata.index.values),:]],
                                                 axis=1).reset_index()
            all_dfs.append(df_both_meta)
            
all_dfs=pd.concat(all_dfs)
all_dfs.head()

In [ ]:
full_dfp7= all_dfs.loc[all_dfs['passage']==7,:]
full_dfp0= all_dfs.loc[all_dfs['passage']==0,:]

full_dfp0.head()

In [ ]:
all_diffs = []
for passage in [1,2,3,4,5,6,7]:
    e003_metadatap7 = e003_metadata.loc[e003_metadata['passage'] == passage,:]
    e003_metadatap7['replicate'] = np.nan 
    e003_metadatap7 = e003_metadatap7.sort_values(by='comm')
    reps = [1,2,3,4]
    full_dfp7= all_dfs.loc[all_dfs['passage']==passage,:]
    full_dfp7['type_meso']=full_dfp7['type_mesocosm'].copy()
    full_dfp7['species-type_meso'] = full_dfp7['species_id'].astype(str) + '-' + full_dfp7['type_meso']
    full_dfp7['replicate'] = 0.
    full_dfp0['type_meso']=full_dfp0['type_mesocosm']
    full_dfp0['species-type_meso'] = full_dfp0['species_id'].astype(str) + '-' + full_dfp0['type_meso']
    full_dfp0med=full_dfp0.groupby(['species-type_meso']).median(numeric_only=True).reset_index()
    full_dfp7med=full_dfp7.groupby(['species-type_meso']).median(numeric_only=True).reset_index()
    full_dfp7=full_dfp7.loc[full_dfp7['species-type_meso'].isin(full_dfp0med['species-type_meso'].values),:]
    
    species_type_mesos=[]
    meso_1s=[]
    meso_2s=[]
    strain_freq_diffs=[]
    med_p0_freqs=[]
    med_p7_freqs = []
    for species_type_meso in full_dfp7['species-type_meso'].unique():
        num_reps = len(full_dfp7.loc[full_dfp7['species-type_meso'] == species_type_meso, 'replicate'])
        full_dfp7.loc[full_dfp7['species-type_meso'] == species_type_meso, 'replicate'] = reps[:num_reps]
      #  print(full_dfp0med.loc[full_dfp0med['species-type_meso'] == species_type_meso, 'actual_med1'].values)
        full_dfp7.loc[full_dfp7['species-type_meso'] == species_type_meso, 'p0_med']= \
            full_dfp0med.loc[full_dfp0med['species-type_meso'] == species_type_meso, 'actual_med1'].values[0]
        full_dfp7_sp_type_meso = full_dfp7.loc[full_dfp7['species-type_meso'] == species_type_meso,:]
        if num_reps>1:
            for meso1,meso2 in it.combinations(full_dfp7_sp_type_meso['mesocosm'].unique(),2):
                species_type_mesos.append(species_type_meso)
                meso_1s.append(meso1)
                meso_2s.append(meso2)
                med_p0_freqs.append(full_dfp0med.loc[full_dfp0med['species-type_meso'] == species_type_meso, 'actual_med1'].values[0])
                med_p7_freqs.append(full_dfp7med.loc[full_dfp7med['species-type_meso'] == species_type_meso, 'actual_med1'].values[0])
                meso1_freq=full_dfp7_sp_type_meso.loc[full_dfp7_sp_type_meso['mesocosm']==meso1,'actual_med1'].values[0]
                meso2_freq=full_dfp7_sp_type_meso.loc[full_dfp7_sp_type_meso['mesocosm']==meso2,'actual_med1'].values[0]
                if meso1 == meso2:
                    print('ck')
                strain_freq_diffs.append(meso1_freq-meso2_freq)
            
    df_diffs = pd.DataFrame(data={'species_type_meso': species_type_mesos,
                                  'Meso1': meso_1s,'Meso2': meso_2s,'strain_freq_diffs':strain_freq_diffs,
                              'p0_freqs':med_p0_freqs, 'p7_freqs': med_p7_freqs})
    df_diffs['strain_freq_diffs_abs']=df_diffs['strain_freq_diffs'].abs()
    df_diffs['passage']=passage
    all_diffs.append(df_diffs)


In [ ]:
all_diffs_df = pd.concat(all_diffs)

all_diffs_df_nonzero = all_diffs_df.loc[(all_diffs_df['p7_freqs']>1e-2)*(all_diffs_df['p7_freqs']<1-1e-2),:]
frequencies, edges = np.histogram(all_diffs_df['strain_freq_diffs_abs'], 20)
    #print('Values: %s, Edges: %s' % (frequencies.shape[0], edges.shape[0]))
hv.Histogram((edges, frequencies)).opts(width=400, height=200, xlabel='|Diff Between Replicates|', 
                                            ylabel='Counts',fill_color= bokeh.palettes.Bright[6][1]
                                           )

In [ ]:
all_diffs_df

In [ ]:
all_dfs=[]
species_list = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/*/')
species_list =[sp.split('/')[-2] for sp in species_list]
e003_metadata = pd.read_csv('e003_coalescence_metadata_round4.csv').set_index('sample')

for sp in species_list:
    for ino in ['AA-AE-mBHI','AA-AF-mBHI','AE-AF-mBHI',
                'AA-AE-mGAM','AA-AF-mGAM','AE-AF-mGAM',]:
     #   try:
        fname1=f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{sp}/{ino}_parent1_info.csv'
        if os.path.exists(fname1):
            
            df_both=get_both_dfs(fname1)
            df_both = df_both.loc[np.intersect1d(df_both.index.values,e003_metadata.index.values),:]
            df_both_meta = pd.concat([df_both,e003_metadata.loc[np.intersect1d(df_both.index.values,
                                                                                   e003_metadata.index.values),:]],
                                         axis=1).reset_index()
       # except:
        #    continue
            df_both_meta['species_id']=sp
            df_both_meta['diff_from_in1']=df_both_meta['actual_med1']
            df_both_meta['diff_from_in2']=df_both_meta['actual_med2']
            in_samples = df_both_meta['inoculumn_sample'].unique()
            if len(in_samples) != len(df_both_meta.loc[df_both_meta['sample'].isin(in_samples),'actual_med1']):
                continue
            
            for in_sample in df_both_meta['inoculumn_sample'].unique():
                
                with_in_sample = df_both_meta.loc[df_both_meta['inoculumn_sample']==in_sample,'sample'].values
                in1_val=df_both_meta.loc[df_both_meta['sample']==in_sample,'actual_med1'].values[0]
                df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in1']= \
                    df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in1'] - in1_val
                
                in2_val=df_both_meta.loc[df_both_meta['sample']==in_sample,'actual_med2'].values[0]
                df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in2']= \
                    df_both_meta.loc[df_both_meta['sample'].isin(with_in_sample),'diff_from_in2'] - in2_val
            all_dfs.append(df_both_meta)
all_dfs=pd.concat(all_dfs)
all_dfs['species-type_meso']=all_dfs['species_id'].astype(str)+'-'+all_dfs['type_mesocosm']
#all_dfs_gr=all_dfs.groupby(['species-type_meso']).median(numeric_only=True).reset_index()

In [ ]:
all_dfs['species-mesocosm']=all_dfs['species_id'].astype(str) + '-' + all_dfs['mesocosm'].astype(str)
all_dfs7 = all_dfs.loc[all_dfs['passage']==7,:]
all_dfs0 = all_dfs.loc[all_dfs['passage']==0,:]
all_dfs7['passage_0'] = np.nan
all_dfs7['sp_ino']=all_dfs7['species_id'].astype(str)+'-'+all_dfs7['inoculumn_sample']
all_dfs0['sp_ino']=all_dfs0['species_id'].astype(str)+'-'+all_dfs0['inoculumn_sample']
for spin in all_dfs7['sp_ino'].unique():
    in_sample = '-'.join(spin.split('-')[1:])
    all_dfs0_good=all_dfs0.loc[all_dfs0['sp_ino']==spin,:]
    all_dfs0_good=all_dfs0_good.loc[all_dfs0_good['sample']==in_sample,'actual_med1'].values
    if len(all_dfs0_good)==0:
        continue
    all_dfs7.loc[all_dfs7['sp_ino']==spin,'passage_0']=all_dfs0_good[0]

    

In [ ]:
all_dfs7 = all_dfs7.loc[~all_dfs7['passage_0'].isna(),:]
all_dfs7p = all_dfs7.copy()
all_dfs7p.loc[all_dfs7p['passage_0']>.5,'actual_med1']=1.-all_dfs7p.loc[all_dfs7p['passage_0']>.5,'actual_med1']
all_dfs7p.loc[all_dfs7p['passage_0']>.5,'passage_0']=1.-all_dfs7p.loc[all_dfs7p['passage_0']>.5,'passage_0']
all_dfs7p.head()

In [ ]:
all_dfs=all_dfs.loc[all_dfs['total_shift']<.1,:]
all_dfs_p7 = all_dfs.loc[all_dfs['passage']==7,:]
all_dfs_p7['diff_from_in1_abs']=all_dfs_p7['diff_from_in1'].abs()
all_dfs_gr=all_dfs_p7.groupby(['species-type_meso']).median(numeric_only=True).reset_index()
all_dfs_gr['diff_from_in1_abs']=all_dfs_gr['diff_from_in1'].abs()
frequencies, edges = np.histogram(all_dfs_p7['diff_from_in1_abs'],bins=20)
#print('Values: %s, Edges: %s' % (frequencies.shape[0], edges.shape[0]))
hv.Histogram((edges, frequencies)).opts(width=400, height=200, xlabel='|Absolute Frequency Shift|', 
                                        ylabel='Counts',fill_color= bokeh.palettes.Bright[6][1]
                                       )

In [ ]:
all_dfs_p7['freq_shift']=all_dfs_p7["diff_from_in1_abs"]
#big_df.loc[big_df['compare_type']=='Btw',:] = 'Over time (P0-7)'
#big_df.loc[big_df['compare_type']=='Wtn',:] = 'Btw Reps'
all_dfs_p7['compare_type']='Over time (P0-7)'
all_diffs_df['freq_shift']=all_diffs_df["strain_freq_diffs_abs"]
all_diffs_df['compare_type']='Btw Reps'

In [ ]:
def ecdf_transform(data):
    return 1- data.rank(method="first") / len(data)

all_dfs_p7.loc[:, "freq shift ECDF"] = all_dfs_p7[
    "freq_shift"
].transform(ecdf_transform)


all_diffs_df.loc[:, "freq shift ECDF"] = all_diffs_df[
    "freq_shift"
].transform(ecdf_transform)
big_df = pd.concat([all_dfs_p7[['freq_shift','compare_type']],
                    all_diffs_df[['freq_shift','compare_type']]])
big_df.loc[:, "freq shift ECDF"] = big_df.groupby('compare_type')[
    "freq_shift"
].transform(ecdf_transform)

In [ ]:

p1=hv.Scatter(
    data=all_dfs_p7,
    kdims='freq_shift',
    vdims=[('freq shift ECDF', 'ECDF'), 'compare_type'],
).opts(color='compare_type')
p2=hv.Scatter(
    data=all_diffs_df,
    kdims='freq_shift',
    vdims=[('freq shift ECDF', 'ECDF'), 'compare_type'],
).opts(color='compare_type')
p1*p2.opts(show_legend=True)

p3=hv.Scatter(
    data=big_df,
    kdims='freq_shift',
    vdims=[('freq shift ECDF', 'ECDF'), 'compare_type'],
).opts(color='compare_type',cmap = [bokeh.palettes.Bright[6][1],bokeh.palettes.Vibrant[7][-1],],
       width=295,height=276,xlabel='Freq Shift',#legend_position='right',
      ylabel='1-CDF')
p3=hv.render(p3)
p3.output_backend = "svg"
export_plot_pdf(p3, 'overtime_btwn')